# S5: LoRA SFT of Tiny Aya En-Thinker on teacher trajectories

**Before running:** Accelerator = GPU T4 x2, Internet = On, secret `HF_TOKEN` attached,
datasets `agentic-distill-src` (code) and `agentic-distill-sft` (train_v2.jsonl) attached.

Set `STAGE` below:
- `"auto"` (default): memory test -> 10-example overfit -> gate -> full 3-epoch run, in one session. Stops at the first failure.
- `"smoke"`: memory test + overfit only.
- `"full"`: skip the checks and train (adapter saved after each epoch to /kaggle/working/out/lora/epochN).

Gate (my proposal, not tuned): memory test ok and peak < 14 GB; overfit last-10-step mean loss < 0.9 x first-10-step mean;
at most 3 skipped fp16 steps; no non-finite loss.


In [ ]:
STAGE = "auto"   # "auto", "smoke" or "full"
import sys, subprocess, os, glob
print(sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
SFT = glob.glob("/kaggle/input/**/train_v2.jsonl", recursive=True)
print("sft:", SFT)
assert SFT, "attach the private agentic-distill-sft dataset"
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # value never printed
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))


In [ ]:
%%bash
set -e
rm -rf /kaggle/working/repo && mkdir -p /kaggle/working/repo
Z=$(find /kaggle/input -name agentic_distill_src.zip | head -1)
if [ -n "$Z" ]; then (cd /kaggle/working/repo && unzip -q "$Z"); \
else D=$(dirname "$(find /kaggle/input -name pyproject.toml | head -1)"); cp -r "$D"/. /kaggle/working/repo/; fi
# Kaggle ships torchao 0.10.0, which this peft rejects at import; we do not use torchao.
pip uninstall -y -q torchao
pip install -q -U transformers peft accelerate safetensors
python -c "import torch, transformers, peft; print('torch', torch.__version__, 'transformers', transformers.__version__, 'peft', peft.__version__, 'gpus', torch.cuda.device_count())"
cd /kaggle/working/repo && PYTHONPATH=src python -m pytest -q tests/test_loss_positions.py 2>&1 | tail -2


In [ ]:
# P1 check: the checkpoint has bias tensors that Cohere2 in transformers has no slot for (load report: UNEXPECTED).
# If they are all ~0, dropping them is harmless; if not, B0 and our fine-tune both differ from the published model.
import json, collections
from huggingface_hub import hf_hub_download
from safetensors import safe_open
MID = "CohereLabs/tiny-aya-en-thinker"
idx = json.load(open(hf_hub_download(MID, "model.safetensors.index.json", token=os.environ["HF_TOKEN"])))["weight_map"]
bias_keys = [k for k in idx if k.endswith(".bias")]
by_file = collections.defaultdict(list)
for k in bias_keys: by_file[idx[k]].append(k)
stats = collections.defaultdict(float)
for f, keys in by_file.items():
    with safe_open(hf_hub_download(MID, f, token=os.environ["HF_TOKEN"]), "pt") as sf:
        for k in keys:
            t = sf.get_tensor(k).float()
            name = k.split(".layers.")[-1].split(".", 1)[-1] if ".layers." in k else k
            stats[name] = max(stats[name], t.abs().max().item())
print("bias tensors:", len(bias_keys))
for n, v in sorted(stats.items()): print(f"  max|bias| {n}: {v:.6g}")


In [ ]:
import subprocess, os, json, statistics as st
SFT_PATH = SFT[0]
env = {**os.environ, "PYTHONPATH": "src", "CUDA_VISIBLE_DEVICES": "0",
       "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
def run(args):
    print(" ".join(args), flush=True)
    p = subprocess.Popen(["python", "-m", "agentic_distill.train_lora", "--data", SFT_PATH] + args,
                         cwd="/kaggle/working/repo", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:                      # no filtering: kernel-selection warnings explain fallbacks
        print(line, end="", flush=True)
    assert p.wait() == 0, f"train_lora exited {p.returncode}"

def smoke():
    run(["--out", "/kaggle/working/out/memtest", "--memtest"])
    run(["--out", "/kaggle/working/out/overfit10", "--limit", "10", "--grad-accum", "1", "--epochs", "4"])

def gate():
    mem = json.load(open("/kaggle/working/out/memtest/result.json"))
    recs = [json.loads(l) for l in open("/kaggle/working/out/overfit10/train_log.jsonl")]
    first, last = st.mean(r["loss"] for r in recs[:10]), st.mean(r["loss"] for r in recs[-10:])
    skipped = sum(r["skipped"] for r in recs)
    peak = max(mem["peak_mem_gb"], max(r["peak_mem_gb"] for r in recs))
    checks = {"memtest_ok": mem["ok"], "peak_mem_gb<14": peak < 14.0,
              "loss_fell_10pct": last < 0.9 * first, "skipped<=3": skipped <= 3,
              "finite": all(r["loss"] == r["loss"] and abs(r["loss"]) < 1e6 for r in recs)}
    print(json.dumps({"first10_loss": round(first, 4), "last10_loss": round(last, 4), "skipped": skipped,
                      "peak_mem_gb": peak, "tokens_per_s_memtest": mem.get("tokens_per_s"), "checks": checks}))
    assert all(checks.values()), f"gate failed: {checks}"

def full():
    run(["--out", "/kaggle/working/out/lora", "--epochs", "3", "--resume"])

if STAGE == "smoke": smoke()
elif STAGE == "auto": smoke(); gate(); full()
else: full()


In [ ]:
import json, glob
for p in glob.glob("/kaggle/working/out/*/train_log.jsonl"):
    recs = [json.loads(l) for l in open(p)]
    print(p, "steps", len(recs), "first loss", recs[0]["loss"], "last loss", recs[-1]["loss"],
          "skipped", sum(r["skipped"] for r in recs), "peak_mem_gb", max(r["peak_mem_gb"] for r in recs))
